### Data Audit & Cleaning

Purpose

This notebook performs a complete audit of the raw shipment  dataset, identifies data quality issues, documents every cleaning decision, and exports a cleaned dataset for downstream analysis.

The objective is to ensure that all business insights are generated from reliable and well-documented data.

## Business Context

The Operations team wants to understand:

- Delivery performance
- Carrier efficiency
- Customer delays
- Freight costs

Before any business conclusions can be drawn, the dataset must be assessed for completeness, accuracy, consistency, and validity.

In [127]:
import pandas as pd
import numpy as np

from IPython.display import display

In [128]:
df = pd.read_csv("../data/shipments.csv")

In [129]:
df.head()

,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status
0,SHP003604,2026-04-18,2026-04-21,2026-04-27,Chandigarh,Guwahati,North,PTL,CARR_07,CUST_085,2026-04-27,2026-04-29,72921.16,894,Delivered
1,SHP003362,2026-06-01,2026-06-01,2026-06-04,Kolkata,Raipur,East,FTL,CARR_03,CUST_108,2026-06-04,2026-06-03,2451.90,91,Delivered
2,SHP000796,2026-01-17,2026-01-20,2026-01-23,Surat,Jaipur,West,FTL,CARR_05,CUST_108,2026-01-23,NaN,10320.66,395,In-Transit
3,SHP000291,2026-04-25,2026-04-25,2026-05-01,Nagpur,Bhubaneswar,Central,FTL,CARR_09,CUST_063,2026-05-01,2026-05-02,30242.52,1223,Delayed
4,SHP003739,2026-01-29,2026-01-30,2026-02-03,Ahmedabad,Hyderabad,West,LTL,CARR_04,CUST_064,2026-02-03,2026-02-01,18434.26,1664,Delivered


In [130]:
df.shape

(5015, 15)

In [131]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5015 entries, 0 to 5014
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   shipment_id             5015 non-null   str    
 1   booking_date            4944 non-null   str    
 2   pickup_date             4927 non-null   str    
 3   delivery_date           5015 non-null   str    
 4   origin_city             5015 non-null   str    
 5   destination_city        5015 non-null   str    
 6   region                  5015 non-null   str    
 7   mode                    5015 non-null   str    
 8   carrier_id              5015 non-null   str    
 9   customer_id             5015 non-null   str    
 10  promised_delivery_date  5015 non-null   str    
 11  actual_delivery_date    3527 non-null   str    
 12  freight_cost            5015 non-null   float64
 13  distance_km             5015 non-null   int64  
 14  status                  5015 non-null   str    
dty

In [132]:
df.describe(include="all")

,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status
count,5015,4944,4927,5015,5015,5015,5015,5015,5015,5015,5015,3527,5015.000000,5015.000000,5015
unique,5000,181,184,189,20,20,5,3,15,120,189,191,NaN,NaN,4
top,SHP000871,2026-05-30,2026-05-30,2026-06-03,Surat,Lucknow,West,FTL,CARR_11,CUST_085,2026-06-03,2026-06-25,NaN,NaN,Delivered
freq,2,46,41,41,279,285,1052,2043,364,60,41,29,NaN,NaN,3616
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33729.236782,1280.035494,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65442.575032,711.756670,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,412.310000,50.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9511.410000,655.500000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18014.500000,1280.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32422.895000,1901.000000,NaN


In [133]:
df.columns

Index(['shipment_id', 'booking_date', 'pickup_date', 'delivery_date',
       'origin_city', 'destination_city', 'region', 'mode', 'carrier_id',
       'customer_id', 'promised_delivery_date', 'actual_delivery_date',
       'freight_cost', 'distance_km', 'status'],
      dtype='str')

In [134]:
missing = df.isnull().sum()

missing

shipment_id                  0
booking_date                71
pickup_date                 88
delivery_date                0
origin_city                  0
destination_city             0
region                       0
mode                         0
carrier_id                   0
customer_id                  0
promised_delivery_date       0
actual_delivery_date      1488
freight_cost                 0
distance_km                  0
status                       0
dtype: int64

In [135]:
missing_percent = (
    df.isnull().sum()
    / len(df)
    *100
).round(2)

missing_percent

shipment_id                0.00
booking_date               1.42
pickup_date                1.75
delivery_date              0.00
origin_city                0.00
destination_city           0.00
region                     0.00
mode                       0.00
carrier_id                 0.00
customer_id                0.00
promised_delivery_date     0.00
actual_delivery_date      29.67
freight_cost               0.00
distance_km                0.00
status                     0.00
dtype: float64

In [136]:
missing_table = pd.DataFrame({

    "Missing Values":missing,

    "Percentage (%)":missing_percent

})

missing_table

,Missing Values,Percentage (%)
shipment_id,0,0.00
booking_date,71,1.42
pickup_date,88,1.75
delivery_date,0,0.00
origin_city,0,0.00
destination_city,0,0.00
region,0,0.00
mode,0,0.00
carrier_id,0,0.00
customer_id,0,0.00


In [137]:
missing_table[
    missing_table["Missing Values"]>0
]

,Missing Values,Percentage (%)
booking_date,71,1.42
pickup_date,88,1.75
actual_delivery_date,1488,29.67


In [138]:
df.duplicated().sum()

np.int64(15)

In [139]:
df[df.duplicated()]

,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status
1363,SHP004419,2026-03-20,2026-03-23,2026-03-27,Kochi,Bhopal,South,FTL,CARR_11,CUST_060,2026-03-27,NaN,5223.17,215,In-Transit
1423,SHP001187,2026-02-27,2026-02-27,2026-03-01,Lucknow,Raipur,North,FTL,CARR_13,CUST_059,2026-03-01,NaN,58368.08,2237,In-Transit
1977,SHP001384,2026-03-24,2026-03-27,2026-03-30,Indore,Nagpur,Central,PTL,CARR_09,CUST_022,2026-03-30,2026-04-04,14829.38,1706,Delivered
2143,SHP000625,2026-05-19,2026-05-22,2026-05-26,Kochi,Ahmedabad,South,FTL,CARR_10,CUST_021,2026-05-26,2026-05-29,47876.69,1841,Delivered
2166,SHP002911,2026-03-03,2026-03-06,2026-03-08,Chandigarh,Mumbai,North,FTL,CARR_08,CUST_067,2026-03-08,2026-03-10,57687.00,2180,Delivered
2636,SHP000280,2026-05-30,NaN,2026-06-09,Delhi,Indore,North,LTL,CARR_09,CUST_118,2026-06-09,2026-06-07,13019.33,1182,Delayed
2842,SHP000871,2026-05-05,2026-05-05,2026-05-07,Chennai,Surat,South,LTL,CARR_14,CUST_021,2026-05-07,NaN,4416.32,324,Delivered
3410,SHP000611,2026-05-09,2026-05-10,2026-05-16,Bengaluru,Guwahati,South,PTL,CARR_09,CUST_013,2026-05-16,NaN,12830.67,1615,Cancelled
3525,SHP002523,2026-04-14,2026-04-14,2026-04-18,Kolkata,Nagpur,East,FTL,CARR_08,CUST_003,2026-04-18,2026-04-21,33966.48,1310,Delivered
3528,SHP002172,2026-03-06,2026-03-07,2026-03-13,Bengaluru,Hyderabad,South,FTL,CARR_14,CUST_097,2026-03-13,NaN,43859.85,1786,Delivered


In [140]:
df["shipment_id"].duplicated().sum()
df[df["shipment_id"].duplicated(keep=False)]

,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status
131,SHP000871,2026-05-05,2026-05-05,2026-05-07,Chennai,Surat,South,LTL,CARR_14,CUST_021,2026-05-07,NaN,4416.32,324,Delivered
237,SHP001349,2026-01-28,2026-01-31,2026-02-08,Bhubaneswar,Bhopal,East,FTL,CARR_01,CUST_039,2026-02-08,2026-02-10,9711.15,355,Delivered
277,SHP000611,2026-05-09,2026-05-10,2026-05-16,Bengaluru,Guwahati,South,PTL,CARR_09,CUST_013,2026-05-16,NaN,12830.67,1615,Cancelled
302,SHP000517,2026-01-03,2026-01-06,2026-01-10,Mumbai,Patna,West,LTL,CARR_03,CUST_040,2026-01-10,2026-01-12,11392.79,973,Delivered
409,SHP004419,2026-03-20,2026-03-23,2026-03-27,Kochi,Bhopal,South,FTL,CARR_11,CUST_060,2026-03-27,NaN,5223.17,215,In-Transit
430,SHP000280,2026-05-30,NaN,2026-06-09,Delhi,Indore,North,LTL,CARR_09,CUST_118,2026-06-09,2026-06-07,13019.33,1182,Delayed
711,SHP001384,2026-03-24,2026-03-27,2026-03-30,Indore,Nagpur,Central,PTL,CARR_09,CUST_022,2026-03-30,2026-04-04,14829.38,1706,Delivered
813,SHP001187,2026-02-27,2026-02-27,2026-03-01,Lucknow,Raipur,North,FTL,CARR_13,CUST_059,2026-03-01,NaN,58368.08,2237,In-Transit
1126,SHP002172,2026-03-06,2026-03-07,2026-03-13,Bengaluru,Hyderabad,South,FTL,CARR_14,CUST_097,2026-03-13,NaN,43859.85,1786,Delivered
1363,SHP004419,2026-03-20,2026-03-23,2026-03-27,Kochi,Bhopal,South,FTL,CARR_11,CUST_060,2026-03-27,NaN,5223.17,215,In-Transit


In [141]:
df.dtypes

shipment_id                   str
booking_date                  str
pickup_date                   str
delivery_date                 str
origin_city                   str
destination_city              str
region                        str
mode                          str
carrier_id                    str
customer_id                   str
promised_delivery_date        str
actual_delivery_date          str
freight_cost              float64
distance_km                 int64
status                        str
dtype: object

In [142]:
(df["freight_cost"]==0).sum()

np.int64(0)

In [143]:
df[df["freight_cost"]<0].sum()

shipment_id                  
booking_date                 
pickup_date                  
delivery_date                
origin_city                  
destination_city             
region                       
mode                         
carrier_id                   
customer_id                  
promised_delivery_date       
actual_delivery_date         
freight_cost              0.0
distance_km                 0
status                       
dtype: object

In [144]:
(df["distance_km"]<0).sum()

np.int64(0)

In [145]:
(df["distance_km"]==0).sum()

np.int64(0)

In [146]:
df["customer_id"].str.strip().eq("").sum()

np.int64(0)

In [147]:
df["carrier_id"].str.strip().eq("").sum()

np.int64(0)

In [148]:
sorted(df["region"].unique())

['Central', 'East', 'North', 'South', 'West']

In [149]:
sorted(df["mode"].unique())

['FTL', 'LTL', 'PTL']

In [150]:
sorted(df["origin_city"].unique())

['Ahmedabad',
 'Bengaluru',
 'Bhopal',
 'Bhubaneswar',
 'Chandigarh',
 'Chennai',
 'Delhi',
 'Guwahati',
 'Hyderabad',
 'Indore',
 'Jaipur',
 'Kochi',
 'Kolkata',
 'Lucknow',
 'Mumbai',
 'Nagpur',
 'Patna',
 'Pune',
 'Raipur',
 'Surat']

In [151]:
sorted(df["destination_city"].unique())

['Ahmedabad',
 'Bengaluru',
 'Bhopal',
 'Bhubaneswar',
 'Chandigarh',
 'Chennai',
 'Delhi',
 'Guwahati',
 'Hyderabad',
 'Indore',
 'Jaipur',
 'Kochi',
 'Kolkata',
 'Lucknow',
 'Mumbai',
 'Nagpur',
 'Patna',
 'Pune',
 'Raipur',
 'Surat']

In [152]:
sorted(df["carrier_id"].unique())

['CARR_01',
 'CARR_02',
 'CARR_03',
 'CARR_04',
 'CARR_05',
 'CARR_06',
 'CARR_07',
 'CARR_08',
 'CARR_09',
 'CARR_10',
 'CARR_11',
 'CARR_12',
 'CARR_13',
 'CARR_14',
 'CARR_15']

In [153]:
df["pickup_date"]

0       2026-04-21
1       2026-06-01
2       2026-01-20
3       2026-04-25
4       2026-01-30
           ...    
5010    2026-02-03
5011    2026-04-06
5012    2026-03-21
5013    2026-06-03
5014    2026-05-08
Name: pickup_date, Length: 5015, dtype: str

In [154]:
df["delivery_date"]

0       2026-04-27
1       2026-06-04
2       2026-01-23
3       2026-05-01
4       2026-02-03
           ...    
5010    2026-02-10
5011    2026-04-09
5012    2026-03-26
5013    2026-06-08
5014    2026-05-15
Name: delivery_date, Length: 5015, dtype: str

In [155]:
import plotly.express as px

In [156]:
fig = px.box(
    df,
    y="freight_cost",
    title="Distribution of Freight Cost"
)

fig.show()

In [157]:
fig = px.box(
    df,
    y="distance_km",
    title="Distribution of Shipment Distance"
)

fig.show()

In [158]:
Q1 = df["freight_cost"].quantile(0.25)
Q3 = df["freight_cost"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5*IQR
upper = Q3 + 1.5*IQR

print(lower)
print(upper)

-24855.8175
66790.1225


In [159]:
freight_outliers = df[
    (df["freight_cost"] < lower) |
    (df["freight_cost"] > upper)
]

freight_outliers

,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status
0,SHP003604,2026-04-18,2026-04-21,2026-04-27,Chandigarh,Guwahati,North,PTL,CARR_07,CUST_085,2026-04-27,2026-04-29,72921.16,894,Delivered
13,SHP004871,2026-04-23,2026-04-24,2026-04-30,Delhi,Nagpur,North,PTL,CARR_07,CUST_055,2026-04-30,2026-04-25,185471.23,2265,Delivered
32,SHP001183,2026-05-25,2026-05-28,2026-06-02,Bhubaneswar,Hyderabad,East,PTL,CARR_07,CUST_048,2026-06-02,2026-06-02,153124.97,1633,Delivered
72,SHP003007,2026-01-06,2026-01-08,2026-01-16,Surat,Chennai,West,LTL,CARR_07,CUST_106,2026-01-16,NaN,285966.88,2427,Cancelled
74,SHP001266,2026-02-02,2026-02-02,2026-02-06,Ahmedabad,Hyderabad,West,PTL,CARR_07,CUST_019,2026-02-06,2026-02-08,194109.32,2496,Delivered
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4902,SHP003692,2026-03-19,2026-03-21,2026-03-25,Mumbai,Bengaluru,West,FTL,CARR_07,CUST_044,2026-03-25,NaN,475544.34,1828,In-Transit
4909,SHP001414,2026-02-14,2026-02-16,2026-02-24,Pune,Nagpur,West,LTL,CARR_07,CUST_071,2026-02-24,2026-02-15,295738.61,2206,Delivered
4953,SHP002177,2026-01-06,2026-01-09,2026-01-13,Pune,Kochi,West,LTL,CARR_07,CUST_067,2026-01-13,2026-01-10,88362.15,789,Delayed
4976,SHP003069,2026-04-14,2026-04-17,2026-04-19,Jaipur,Nagpur,North,PTL,CARR_07,CUST_067,2026-04-19,2026-04-26,204282.50,2270,Delivered


In [160]:
print(len(freight_outliers))

296


In [161]:
round(
len(freight_outliers)
/
len(df)
*100,
2)

5.9

In [162]:
Q1 = df["distance_km"].quantile(.25)

Q3 = df["distance_km"].quantile(.75)

IQR = Q3-Q1

lower = Q1-1.5*IQR

upper = Q3+1.5*IQR

print(lower)
print(upper)

-1212.75
3769.25


In [163]:
distance_outliers = df[
(df["distance_km"]<lower) |
(df["distance_km"]>upper)
]

distance_outliers


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status


In [164]:
fig = px.scatter(
    df,
    x="distance_km",
    y="freight_cost",
    title="Freight Cost vs Shipment Distance",
    opacity=0.6
)

fig.show()

### Observations

The freight cost distribution is highly right-skewed, with a relatively small number of shipments exhibiting exceptionally high freight costs.

Using the Interquartile Range (IQR) method, 296 shipments (5.90% of the dataset) were identified as potential freight cost outliers.

These observations are not removed at this stage because high freight costs may be legitimate and could be explained by factors such as:
- Longer shipment distances
- Premium transportation modes
- Customer-specific pricing
- Operational surcharges

The shipment distance distribution does not contain any statistical outliers using the IQR method. Although shipment distances range from 50 km to 2,499 km, all observations fall within the expected range according to the IQR thresholds.

The scatter plot of Freight Cost versus Shipment Distance shows a clear positive relationship between shipment distance and freight cost. Most shipments follow a reasonable upward trend, indicating that freight charges generally increase with distance.

However, several shipments exhibit substantially higher freight costs than other shipments travelling similar distances. These observations should be retained for further investigation during the carrier performance analysis, as they may represent genuine pricing differences rather than data quality issues.

## Business Rule Validation

Beyond completeness and consistency, shipment records should also follow expected operational business rules.

This section validates whether shipment dates occur in a logical sequence and whether shipment statuses are consistent with delivery information.

The objective is to distinguish genuine operational scenarios from potential data quality issues before performing any cleaning.

In [165]:
#Booking date =< picup_date
booking_after_pickup = df[
    pd.to_datetime(df["booking_date"]) >
    pd.to_datetime(df["pickup_date"])
]

print("Violations:", len(booking_after_pickup))
booking_after_pickup.head()

Violations: 0


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status


In [166]:
#pickup_date<= delivery_date
pickup_after_delivery = df[
    pd.to_datetime(df["pickup_date"]) >
    pd.to_datetime(df["delivery_date"])
]

print("Violations:", len(pickup_after_delivery))
pickup_after_delivery.head()

Violations: 0


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status


In [167]:
#Promised Delivery ≥ Booking Date
promise_before_booking = df[
    pd.to_datetime(df["promised_delivery_date"]) <
    pd.to_datetime(df["booking_date"])
]

print("Violations:", len(promise_before_booking))
promise_before_booking.head()

Violations:

 0


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status


In [168]:
#actual delivery date before booking date
actual_before_booking = df[
    pd.to_datetime(df["actual_delivery_date"]) <
    pd.to_datetime(df["booking_date"])
]

print("Violations:", len(actual_before_booking))
actual_before_booking.head()

Violations: 39


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status
37,SHP004729,2026-04-05,2026-04-07,2026-04-13,Chandigarh,Chandigarh,North,FTL,CARR_06,CUST_029,2026-04-13,2026-04-04,34692.56,1527,Delivered
127,SHP002004,2026-01-22,2026-01-23,2026-01-30,Nagpur,Indore,Central,FTL,CARR_11,CUST_099,2026-01-30,2026-01-21,57091.76,2044,Delivered
133,SHP003752,2026-01-26,NaN,2026-01-30,Kolkata,Chennai,East,LTL,CARR_10,CUST_007,2026-01-30,2026-01-23,8631.21,661,Delivered
188,SHP002186,2026-03-27,2026-03-29,2026-03-31,Raipur,Chennai,Central,FTL,CARR_07,CUST_094,2026-03-31,2026-03-26,280439.68,1012,Delivered
369,SHP003624,2026-01-09,2026-01-09,2026-01-12,Guwahati,Raipur,East,PTL,CARR_13,CUST_012,2026-01-12,2026-01-08,20327.97,2249,Delivered


In [169]:
#actual delivery before pickup
actual_before_pickup = df[
    pd.to_datetime(df["actual_delivery_date"]) <
    pd.to_datetime(df["pickup_date"])
]

print("Violations:", len(actual_before_pickup))
actual_before_pickup.head()

Violations: 72


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status
37,SHP004729,2026-04-05,2026-04-07,2026-04-13,Chandigarh,Chandigarh,North,FTL,CARR_06,CUST_029,2026-04-13,2026-04-04,34692.56,1527,Delivered
88,SHP000021,2026-02-03,2026-02-05,2026-02-12,Delhi,Ahmedabad,North,FTL,CARR_06,CUST_099,2026-02-12,2026-02-04,47195.55,1859,Delivered
112,SHP002526,2026-05-21,2026-05-24,2026-05-31,Bhubaneswar,Bengaluru,East,FTL,CARR_14,CUST_116,2026-05-31,2026-05-21,20068.24,728,Delivered
127,SHP002004,2026-01-22,2026-01-23,2026-01-30,Nagpur,Indore,Central,FTL,CARR_11,CUST_099,2026-01-30,2026-01-21,57091.76,2044,Delivered
188,SHP002186,2026-03-27,2026-03-29,2026-03-31,Raipur,Chennai,Central,FTL,CARR_07,CUST_094,2026-03-31,2026-03-26,280439.68,1012,Delivered


In [170]:
#delivered shipment missing actual_delivery_date
delivered_missing_actual = df[
    (df["status"] == "Delivered") &
    (df["actual_delivery_date"].isna())
]

print("Delivered with missing actual delivery:", len(delivered_missing_actual))
delivered_missing_actual.head()

Delivered with missing actual delivery: 588


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status
5,SHP002757,2026-01-05,2026-01-07,2026-01-14,Bengaluru,Bengaluru,South,LTL,CARR_02,CUST_058,2026-01-14,NaN,14006.87,1051,Delivered
17,SHP001870,2026-02-14,2026-02-16,2026-02-23,Hyderabad,Indore,South,FTL,CARR_13,CUST_001,2026-02-23,NaN,62852.46,2464,Delivered
19,SHP003020,2026-06-23,2026-06-24,2026-07-02,Hyderabad,Bengaluru,South,FTL,CARR_13,CUST_066,2026-07-02,NaN,62094.53,2261,Delivered
39,SHP004849,2026-04-30,2026-04-30,2026-05-08,Kochi,Guwahati,South,FTL,CARR_15,CUST_006,2026-05-08,NaN,33423.87,1508,Delivered
46,SHP004215,2026-04-06,2026-04-08,2026-04-10,Chennai,Guwahati,South,FTL,CARR_11,CUST_084,2026-04-10,NaN,21221.44,893,Delivered


In [171]:
#In-Transit Shipments Having Actual Delivery Date
intransit_with_actual = df[
    (df["status"] == "In-Transit") &
    (df["actual_delivery_date"].notna())
]

print("In-Transit with actual delivery:", len(intransit_with_actual))
intransit_with_actual.head()

In-Transit with actual delivery: 0


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status


In [172]:
#cancelled shipments having actual delivery dates
cancelled_with_actual = df[
    (df["status"] == "Cancelled") &
    (df["actual_delivery_date"].notna())
]

print("Cancelled with actual delivery:", len(cancelled_with_actual))
cancelled_with_actual.head()

Cancelled with actual delivery: 0


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status


In [173]:
status_summary = (
    df.groupby("status")["actual_delivery_date"]
      .agg(
          Total_Shipments="size",
          Missing_Actual=lambda x: x.isna().sum()
      )
)

status_summary["Missing_%"] = (
    status_summary["Missing_Actual"] /
    status_summary["Total_Shipments"] * 100
).round(2)

status_summary

,Total_Shipments,Missing_Actual,Missing_%
status,,,
Cancelled,302,302,100.00
Delayed,595,96,16.13
Delivered,3616,588,16.26
In-Transit,502,502,100.00


In [174]:
status_distribution = (
    df["status"]
      .value_counts()
      .to_frame("Shipment_Count")
)

status_distribution["Percentage"] = (
    status_distribution["Shipment_Count"] /
    len(df) * 100
).round(2)

status_distribution

,Shipment_Count,Percentage
status,,
Delivered,3616,72.10
Delayed,595,11.86
In-Transit,502,10.01
Cancelled,302,6.02


# Operational Status vs SLA Validation using the status column

In [175]:
date_columns = [
    "booking_date",
    "pickup_date",
    "delivery_date",
    "promised_delivery_date",
    "actual_delivery_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col])

In [176]:
df[date_columns].dtypes

booking_date              datetime64[us]
pickup_date               datetime64[us]
delivery_date             datetime64[us]
promised_delivery_date    datetime64[us]
actual_delivery_date      datetime64[us]
dtype: object

In [177]:
# Create temporary dataframe for validation
audit_df = df.drop_duplicates().copy()

print("Original records:", len(df))
print("Unique records:", len(audit_df))

Original records: 5015
Unique records: 5000


In [178]:
# creating label manually for verification
audit_df["expected_status"] = audit_df["status"]  # temporary initialization

In [179]:
audit_df.loc[
    audit_df["actual_delivery_date"].notna() &
    (
        audit_df["actual_delivery_date"]
        <=
        audit_df["promised_delivery_date"]
    ),
    "expected_status"
] = "Delivered"

In [180]:
audit_df.loc[
    audit_df["actual_delivery_date"].notna() &
    (
        audit_df["actual_delivery_date"]
        >
        audit_df["promised_delivery_date"]
    ),
    "expected_status"
] = "Delayed"

In [181]:
comparison = pd.crosstab(

    audit_df["status"],

    audit_df["expected_status"],

    margins=True

)

comparison

expected_status,Cancelled,Delayed,Delivered,In-Transit,All
status,,,,,
Cancelled,301,0,0,0,301
Delayed,0,335,258,0,593
Delivered,0,1484,2123,0,3607
In-Transit,0,0,0,499,499
All,301,1819,2381,499,5000


In [182]:
status_mismatch = (

    audit_df["status"]

    !=

    audit_df["expected_status"]

).sum()

status_accuracy = (

    1 -

    status_mismatch

    /

    len(audit_df)

) * 100

print(f"Status mismatches: {status_mismatch}")

print(f"Status Accuracy: {status_accuracy:.2f}%")

Status mismatches: 1742
Status Accuracy: 65.16%


In [183]:
mismatch_df = audit_df[
    audit_df["status"] != audit_df["expected_status"]
]

mismatch_df.head(20)

,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status,expected_status
0,SHP003604,2026-04-18,2026-04-21,2026-04-27,Chandigarh,Guwahati,North,PTL,CARR_07,CUST_085,2026-04-27,2026-04-29,72921.16,894,Delivered,Delayed
6,SHP004951,2026-04-07,2026-04-10,2026-04-12,Bhubaneswar,Kolkata,East,LTL,CARR_05,CUST_030,2026-04-12,2026-04-20,14556.56,1195,Delivered,Delayed
8,SHP003173,2026-06-27,2026-06-27,2026-07-03,Bhubaneswar,Chennai,East,FTL,CARR_13,CUST_022,2026-07-03,2026-07-03,41723.37,1798,Delayed,Delivered
11,SHP000850,2026-04-29,2026-05-01,2026-05-05,Surat,Surat,West,LTL,CARR_08,CUST_050,2026-05-05,2026-05-07,17973.12,1683,Delivered,Delayed
12,SHP000876,2026-06-08,2026-06-11,2026-06-14,Lucknow,Chennai,North,LTL,CARR_06,CUST_046,2026-06-14,2026-06-20,15922.55,1167,Delivered,Delayed
14,SHP004001,2026-04-30,2026-05-03,2026-05-10,Mumbai,Patna,West,FTL,CARR_12,CUST_057,2026-05-10,2026-05-13,42573.77,1714,Delivered,Delayed
16,SHP003881,2026-03-17,2026-03-20,2026-03-25,Pune,Patna,West,FTL,CARR_09,CUST_066,2026-03-25,2026-03-21,29445.67,1127,Delayed,Delivered
20,SHP003753,NaT,2026-04-09,2026-04-14,Surat,Raipur,West,FTL,CARR_09,CUST_023,2026-04-14,2026-04-11,41680.29,1796,Delayed,Delivered
23,SHP000666,2026-02-27,2026-02-28,2026-03-07,Bhopal,Chennai,Central,LTL,CARR_08,CUST_018,2026-03-07,2026-03-05,16919.73,1323,Delayed,Delivered
30,SHP001788,2026-01-28,2026-01-28,2026-02-04,Indore,Guwahati,Central,LTL,CARR_06,CUST_032,2026-02-04,2026-02-07,28396.38,2181,Delivered,Delayed


In [184]:
delivered_but_late = audit_df[
    (audit_df["status"] == "Delivered") &
    (audit_df["expected_status"] == "Delayed")
]

print("Delivered but actually late:", len(delivered_but_late))

Delivered but actually late: 1484


In [185]:
delayed_but_ontime = audit_df[
    (audit_df["status"] == "Delayed") &
    (audit_df["expected_status"] == "Delivered")
]

print("Delayed but actually on time:", len(delayed_but_ontime))

Delayed but actually on time: 258


In [186]:
delivered_missing_actual = audit_df[
    (audit_df["status"] == "Delivered") &
    (audit_df["actual_delivery_date"].isna())
]

print(
    "Delivered but missing actual delivery:",
    len(delivered_missing_actual)
)

Delivered but missing actual delivery: 586


In [187]:
delayed_missing_actual = audit_df[
    (audit_df["status"] == "Delayed") &
    (audit_df["actual_delivery_date"].isna())
]

print(
    "Delayed but missing actual delivery:",
    len(delayed_missing_actual)
)

Delayed but missing actual delivery: 96


In [188]:
actual_before_booking.groupby("status").size()

status
Delayed       3
Delivered    36
dtype: int64

In [189]:
actual_before_pickup.groupby("status").size()

status
Delayed       5
Delivered    67
dtype: int64

# Data Cleaning


### Cleaning Decision

Fifteen duplicate shipment records were identified during the audit.

Since each duplicate represented an exact copy of an existing shipment, duplicate rows were removed to prevent double-counting during performance analysis.

In [190]:
clean_df = df.copy()

print(clean_df.shape)

(5015, 15)


In [191]:
rows_before = len(clean_df)

clean_df = clean_df.drop_duplicates()

rows_after = len(clean_df)

print("Rows removed:", rows_before - rows_after)
print("Remaining rows:", rows_after)

Rows removed: 15
Remaining rows: 5000


### Cleaning Decision

All shipment date columns were converted from string format to datetime format to enable time-based calculations such as transit time and delivery delay.

In [192]:
date_columns = [
    "booking_date",
    "pickup_date",
    "delivery_date",
    "promised_delivery_date",
    "actual_delivery_date"
]

for col in date_columns:
    clean_df[col] = pd.to_datetime(clean_df[col])

In [193]:
clean_df[date_columns].dtypes

booking_date              datetime64[us]
pickup_date               datetime64[us]
delivery_date             datetime64[us]
promised_delivery_date    datetime64[us]
actual_delivery_date      datetime64[us]
dtype: object

Rather than deleting records with impossible date sequences, a validation flag was created.

This preserves the original data while allowing analytical calculations to exclude invalid observations when necessary.

In [194]:
clean_df["missing_booking_date"] = clean_df["booking_date"].isna()

clean_df["missing_pickup_date"] = clean_df["pickup_date"].isna()

clean_df["missing_actual_delivery"] = clean_df["actual_delivery_date"].isna()

In [195]:
clean_df["invalid_date_sequence"] = (

    (clean_df["actual_delivery_date"] < clean_df["booking_date"])

    |

    (clean_df["actual_delivery_date"] < clean_df["pickup_date"])

)

In [196]:
clean_df["invalid_date_sequence"].value_counts()

invalid_date_sequence
False    4926
True       74
Name: count, dtype: int64

### Feautre Engineering

Creating important business metrics for further analysis


In [197]:
clean_df["delivery_delay_days"] = (

    clean_df["actual_delivery_date"]

    -

    clean_df["promised_delivery_date"]

).dt.days

In [198]:
clean_df["transit_days"] = (

    clean_df["actual_delivery_date"]

    -

    clean_df["pickup_date"]

).dt.days

In [199]:
clean_df["booking_to_pickup_days"] = (

    clean_df["pickup_date"]

    -

    clean_df["booking_date"]

).dt.days

In [200]:
clean_df["cost_per_km"] = (

    clean_df["freight_cost"]

    /

    clean_df["distance_km"]

)

In [201]:
clean_df["delivery_performance"] = pd.NA

In [202]:
clean_df.loc[
    clean_df["status"] == "Cancelled",
    "delivery_performance"
] = "Cancelled"

In [203]:
clean_df.loc[
    clean_df["status"] == "In-Transit",
    "delivery_performance"
] = "In Transit"

In [204]:
clean_df.loc[
    (
        clean_df["actual_delivery_date"].notna()
    )
    &
    (
        clean_df["actual_delivery_date"]
        <=
        clean_df["promised_delivery_date"]
    ),
    "delivery_performance"
] = "On Time"

In [205]:
clean_df.loc[
    (
        clean_df["actual_delivery_date"].notna()
    )
    &
    (
        clean_df["actual_delivery_date"]
        >
        clean_df["promised_delivery_date"]
    ),
    "delivery_performance"
] = "Late"


In [206]:
clean_df["delivery_performance"] = clean_df["delivery_performance"].fillna("Unknown")

In [207]:
clean_df["on_time_flag"] = np.where(
    clean_df["delivery_performance"] == "On Time",
    1,
    np.where(
        clean_df["delivery_performance"] == "Late",
        0,
        np.nan
    )
)

In [208]:
clean_df["analysis_ready"] = (
    clean_df["actual_delivery_date"].notna()
 ) & (
     ~clean_df["invalid_date_sequence"]
     ) & (clean_df["delivery_performance"] != "Unknown")


In [209]:
clean_df["delay_category"] = pd.cut(

    clean_df["delivery_delay_days"],

    bins=[-1000,-1,0,2,5,1000],

    labels=[
        "Early",

        "On Time",

        "1–2 Days Late",

        "3–5 Days Late",

        ">5 Days Late"
    ]
)

In [210]:
clean_df["distance_band"] = pd.cut(

    clean_df["distance_km"],

    bins=[0,500,1000,1500,2000,2500],

    labels=[
        "0–500 km",

        "501–1000 km",

        "1001–1500 km",

        "1501–2000 km",

        ">2000 km"
    ]
)

## Final Quality Check

In [211]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 5000 entries, 0 to 5014
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   shipment_id              5000 non-null   str           
 1   booking_date             4929 non-null   datetime64[us]
 2   pickup_date              4913 non-null   datetime64[us]
 3   delivery_date            5000 non-null   datetime64[us]
 4   origin_city              5000 non-null   str           
 5   destination_city         5000 non-null   str           
 6   region                   5000 non-null   str           
 7   mode                     5000 non-null   str           
 8   carrier_id               5000 non-null   str           
 9   customer_id              5000 non-null   str           
 10  promised_delivery_date   5000 non-null   datetime64[us]
 11  actual_delivery_date     3518 non-null   datetime64[us]
 12  freight_cost             5000 non-null   float64  

In [212]:
print(clean_df.shape)

(5000, 28)


In [213]:
print(clean_df.duplicated().sum())

0


In [214]:
print(clean_df.isna().sum())

shipment_id                   0
booking_date                 71
pickup_date                  87
delivery_date                 0
origin_city                   0
destination_city              0
region                        0
mode                          0
carrier_id                    0
customer_id                   0
promised_delivery_date        0
actual_delivery_date       1482
freight_cost                  0
distance_km                   0
status                        0
missing_booking_date          0
missing_pickup_date           0
missing_actual_delivery       0
invalid_date_sequence         0
delivery_delay_days        1482
transit_days               1545
booking_to_pickup_days      155
cost_per_km                   0
delivery_performance          0
on_time_flag               1482
analysis_ready                0
delay_category             1482
distance_band                 0
dtype: int64


In [ ]:
clean_df.columns

Index(['shipment_id', 'booking_date', 'pickup_date', 'delivery_date',
       'origin_city', 'destination_city', 'region', 'mode', 'carrier_id',
       'customer_id', 'promised_delivery_date', 'actual_delivery_date',
       'freight_cost', 'distance_km', 'status', 'missing_booking_date',
       'missing_pickup_date', 'missing_actual_delivery',
       'invalid_date_sequence', 'delivery_delay_days', 'transit_days',
       'booking_to_pickup_days', 'cost_per_km', 'delivery_performance',
       'on_time_flag', 'analysis_ready', 'delay_category', 'distance_band'],
      dtype='str')

In [219]:
# Export cleaned dataset
clean_df.to_csv(
    "../Data/shipments_cleaned.csv",
    index=False
)

print("✅ Export successful!")

✅ Export successful!


# Final Dataset Ready for Analysis

The shipment dataset has been audited, cleaned, and transformed into an analytics-ready dataset.

### Cleaning Summary

- Removed 15 duplicate shipment records.
- Converted all date columns to datetime format.
- Preserved missing values where business events could not be reliably inferred.
- Flagged invalid date sequences instead of removing records.
- Added analytical features including delivery delay, transit time, booking lead time, cost per kilometer, and delivery performance.
- Created audit flags to support transparent downstream filtering.

The cleaned dataset has been exported and will serve as the input for the exploratory data analysis and dashboard development notebook.